# 08｜异步任务、诊断与标准错误


> 运行方式：从上到下执行代码单元。控制类 Notebook **没有布尔安全开关**；执行控制单元就会把任务提交给 SDK。
> 需要仿真的课程请先在 FleetDock IDEA 中启动仿真，并确认 `drone1`、`drone2` 等设备在线。


![任务与诊断](assets/task-diagnostics.svg)


## 先诊断，再执行

`diagnostics()` 查看产品、能力和生命周期；`readiness_for()` 判断某个动作当前是否具备条件。


In [ ]:
from fleetdock_runtime import FleetDock
from fleetdock import CapabilityName, Route

FleetDock.init()
drone = FleetDock.devices.drone1

print("诊断：", drone.diagnostics())
print("连接：", drone.connect())
print("起飞就绪：", drone.readiness_for(CapabilityName.TAKEOFF))


## 后台执行

`start()` 返回 `TaskSession`。Notebook 可以继续做别的事情，之后再用 `wait()` 取得标准任务结果。


In [ ]:
route = Route.circle(radius_m=5.0, segments=12, name="异步圆形任务")
session = (
    FleetDock.core.plan(drone)
    .wait_until_ready(timeout_s=60.0)
    .unlock()
    .takeoff(altitude=10.0)
    .route(route, altitude=10.0, speed=2.0)
    .return_to_launch()
    .start(step_timeout_s=180.0)
)

print("会话状态：", session.state)
result = session.wait(timeout_s=240.0)
print("最终结果：", result)


需要中止后台任务时调用 `session.cancel()`；SDK 会按 `TaskCancelMode.SAFE_STOP` 处理取消，而不是让应用直接终止线程。


In [ ]:
bad_response = drone.takeoff(altitude=-1.0)
print("标准错误：", bad_response.ok, bad_response.code, bad_response.message)

FleetDock.close()


## 重点

能力不足、未就绪、参数错误和任务失败都通过统一模型返回，业务代码不需要解析厂商异常文本。
